In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as offline

from scipy.optimize import minimize

In [ ]:
offline.init_notebook_mode(connected=True)
pio.kaleido.scope.default_format = 'pdf'

In [3]:
def pl_loglik_and_grad(xi, rankings, models, ref_model):
    free_models = [m for m in models if m != ref_model]
    idx = {m:i for i,m in enumerate(free_models)}
    theta = {m: np.exp(xi[m]) for m in models}
    loglik = 0.0
    grad = np.zeros(len(free_models), dtype=float)
    for r in rankings:
        S = r[:]
        denom = sum(theta[m] for m in S)
        loglik += np.log(theta[r[0]]) - np.log(denom)
        for m in S:
            if m != ref_model:
                if m == r[0]:
                    grad[idx[m]] += 1.0
                grad[idx[m]] -= (theta[m] / denom)
        S = S[1:]
        denom = sum(theta[m] for m in S)
        loglik += np.log(theta[r[1]]) - np.log(denom)
        for m in S:
            if m != ref_model:
                if m == r[1]:
                    grad[idx[m]] += 1.0
                grad[idx[m]] -= (theta[m] / denom)
        S = S[1:]
        denom = sum(theta[m] for m in S)
        loglik += np.log(theta[r[2]]) - np.log(denom)
        for m in S:
            if m != ref_model:
                if m == r[2]:
                    grad[idx[m]] += 1.0
                grad[idx[m]] -= (theta[m] / denom)
    return loglik, grad


def pack_xi_vector(xi_dict, models, ref_model):
    free_models = [m for m in models if m != ref_model]
    return np.array([xi_dict[m] for m in free_models], dtype=float)


def unpack_xi_vector(x, models, ref_model):
    xi = {}
    for m in models:
        if m == ref_model:
            xi[m] = 0.0
        else:
            # find index
            free_models = [mm for mm in models if mm != ref_model]
            xi[m] = x[free_models.index(m)]
    return xi


def neg_pl_objective(x, rankings, models, ref_model):
    xi = unpack_xi_vector(x, models, ref_model)
    ll, grad = pl_loglik_and_grad(xi, rankings, models, ref_model)
    return -ll, -grad


def hessian_approx(func, x, eps=1e-5):
    n = len(x)
    H = np.zeros((n, n))
    base_val, base_grad = func(x)
    for i in range(n):
        x1 = x.copy(); x1[i] += eps
        _, g1 = func(x1)
        x2 = x.copy(); x2[i] -= eps
        _, g2 = func(x2)
        H[:, i] = (g1 - g2) / (2*eps)
    return H


def calculate_pl_elo(input_file="score.tsv", save_csv=False):
    df = pd.read_csv(input_file, sep='\t')
    model_cols = ['Gemini', 'Grok', 'OpenAI', 'Phytomni']
    for col in model_cols:
        if col not in df.columns:
            raise ValueError(f"Missing column: {col}")
    rankings = []
    for _, row in df.iterrows():
        rank_map = {}
        for m in model_cols:
            val = row[m]
            if isinstance(val, str) and val.startswith('R'):
                try:
                    rnum = int(val[1:])
                except:
                    continue
                rank_map[m] = rnum
        if len(rank_map) != 4:
            continue
        ordered = sorted(rank_map.items(), key=lambda kv: kv[1])
        rankings.append([m for m, r in ordered])
    models = model_cols[:]
    models.sort()
    ref_model = models[0]
    print(f"Models: {models} | Reference model fixed at 0: {ref_model}")
    print(f"Total 4-way battles (rows used): {len(rankings)}")
    xi0 = {m: 0.0 for m in models}
    x0 = pack_xi_vector(xi0, models, ref_model)

    def f_and_g(x):
        return neg_pl_objective(x, rankings, models, ref_model)

    res = minimize(lambda z: f_and_g(z)[0], x0, jac=lambda z: f_and_g(z)[1],
                   method='L-BFGS-B', options={'ftol':1e-10, 'gtol':1e-8, 'maxiter':1000})
    if not res.success:
        print("Warning: optimization did not fully converge:", res.message)
    x_hat = res.x
    xi_hat = unpack_xi_vector(x_hat, models, ref_model)
    H = hessian_approx(lambda z: neg_pl_objective(z, rankings, models, ref_model), x_hat)
    try:
        cov_free = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        print("Hessian not invertible; using pseudo-inverse.")
        cov_free = np.linalg.pinv(H)
    free_models = [m for m in models if m != ref_model]
    cov_full = np.zeros((len(models), len(models)))
    full_idx = {m:i for i,m in enumerate(models)}
    for i,m1 in enumerate(free_models):
        for j,m2 in enumerate(free_models):
            cov_full[full_idx[m1], full_idx[m2]] = cov_free[i,j]
    A = 400.0 / np.log(10.0)
    xi_vec = np.array([xi_hat[m] for m in models])
    Elo_raw = A * xi_vec
    B = 1500.0 - np.mean(Elo_raw)
    Elo = Elo_raw + B
    se_xi = np.sqrt(np.diag(cov_full))
    se_Elo = A * se_xi
    Elo_L = Elo - 1.96 * se_Elo
    Elo_U = Elo + 1.96 * se_Elo
    print("\n=== Elo-like scores from 4-way Plackett–Luce ===")
    out_rows = []
    for m in models:
        i = full_idx[m]
        out_rows.append((m, Elo[i], Elo_L[i], Elo_U[i]))
    out_rows.sort(key=lambda x: x[1], reverse=True)
    for m, e, l, u in out_rows:
        print(f"{m:10s}  Elo={e:7.2f}  95% CI [{l:7.2f}, {u:7.2f}]")
    print("\n=== Pairwise win probability matrix (p(row > col)) ===")
    pmat = np.zeros((len(models), len(models)))
    for i, mi in enumerate(models):
        for j, mj in enumerate(models):
            if i == j:
                pmat[i,j] = np.nan
            else:
                pmat[i,j] = 1.0 / (1.0 + np.exp(xi_hat[mj] - xi_hat[mi]))
    header = "          " + "".join([f"{m:>12s}" for m in models])
    print(header)
    for i, mi in enumerate(models):
        row_str = f"{mi:10s}" + "".join([f"{pmat[i,j]:12.3f}" if i!=j else f"{'--':>12s}" for j in range(len(models))])
        print(row_str)
    if save_csv:
        results_df = pd.DataFrame({
            'Model': [r[0] for r in out_rows],
            'Elo':   [r[1] for r in out_rows],
            'Elo_L': [r[2] for r in out_rows],
            'Elo_U': [r[3] for r in out_rows],
        })
        results_df.to_csv("pl_elo_results.csv", index=False)
        print("Results saved to pl_elo_results.csv")
        df_p = pd.DataFrame(pmat, index=models, columns=models)
        df_p.to_csv("pl_pairwise_probs.csv")
        print("Pairwise probabilities saved to pl_pairwise_probs.csv")
    return out_rows, pmat, xi_hat

In [5]:
model_list = ['Phytomni', 'Gemini', 'OpenAI', 'Grok']
x_list = ['Phytomni', 'Gemini Deep Research', 'ChatGPT Agent mode', 'Grok DeepSearch']

### Fig. 2d

In [6]:
model_rank_list = [[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
with open(f'PhytoBench-Gene-for_plot/score.tsv', 'r') as open_tsv:
    for eachline in open_tsv.readlines()[1:]:
        sp = eachline.strip().split()
        rank_array = np.array([int(s[1]) - 1 for s in sp[-4:]])[[3, 0, 2, 1]]
        for i, r in enumerate(rank_array):
            model_rank_list[i][r] += 1
num = sum(model_rank_list[0])
if sum(model_rank_list[1]) == num and sum(model_rank_list[2]) == num and sum(model_rank_list[3]) == num:
    model_rank_array = np.array(model_rank_list) / num
else:
    raise 'error'

line_width = 2
rank_color_list = ['rgb(0,152,210)', 'rgb(140,197,240)', 'rgb(197,224,247)', 'rgb(230,242,250)']
fig = go.Figure()
for i in list(range(4)):
    fig.add_trace(go.Bar(
        x=x_list,
        y=model_rank_array[:,i],
        name=f'Rank{i+1}',
        marker_color=rank_color_list[i],
    ))
fig.update_layout(
    barmode='stack',
    showlegend=False,
    xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
    yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Percent (%)'},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=1080,
    height=1080)
fig.show()
# file_prefix = f'../Fig. 2/fig.2d.phytobench-gene.percent.bar'.lower().replace(' ', '_')
# fig.write_image(f'{file_prefix}.pdf')
# fig.write_image(f'{file_prefix}.png')

### Fig. 2e

In [ ]:
results, prob_matrix, strengths = calculate_pl_elo(f'PhytoBench-Gene-for_plot/score.tsv', save_csv=False)

reordered_matrix = prob_matrix[np.ix_([3, 0, 2, 1], [3, 0, 2, 1])]
fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=x_list,
    y=x_list,
    z=reordered_matrix,
    colorscale='TroPic',
))
fig.update_layout(
    showlegend=False,
    yaxis_scaleanchor='x',
    xaxis={'showline': False, 'linewidth': line_width, 'linecolor': 'black', 'mirror': False, 'ticks': 'outside', 'tickwidth': line_width},
    yaxis={'showline': False, 'linewidth': line_width, 'linecolor': 'black', 'mirror': False, 'ticks': 'outside', 'tickwidth': line_width},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=1080,
    height=1080)
fig.show()
# file_prefix = f'../Fig. 2/fig.2e.phytobench-gene.{score_str}.prob.heatmap'.lower().replace(' ', '_')
# fig.write_image(f'{file_prefix}.pdf')
# fig.write_image(f'{file_prefix}.png')

Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 600

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1650.36  95% CI [1625.25, 1675.46]
Gemini      Elo=1581.77  95% CI [1581.77, 1581.77]
OpenAI      Elo=1466.78  95% CI [1441.42, 1492.14]
Grok        Elo=1301.09  95% CI [1270.79, 1331.39]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.834       0.660       0.403
Grok             0.166          --       0.278       0.118
OpenAI           0.340       0.722          --       0.258
Phytomni         0.597       0.882       0.742          --


### Fig. 2f

In [ ]:
y_list = [0, 0, 0, 0]
for r in results:
    index = model_list.index(r[0])
    y_list[index] = r[1]

color_list = ['rgb(31,113,179)', 'rgb(208,210,211)', 'rgb(208,210,211)', 'rgb(208,210,211)']
fig = go.Figure()
fig.add_trace(go.Bar(
    x=x_list,
    y=y_list,
    marker_color=color_list,
    marker_line_color='rgb(0,0,0)',
    marker_line_width=line_width,
    text=[f'{round(num)}' for num in y_list],
    textposition='outside',
))
fig.update_layout(
    showlegend=False,
    xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
    yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Score', 'range': [1000, 2000]},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=1080,
    height=1080)
fig.show()
# file_prefix = f'../Fig. 2/fig.2f.phytobench-gene.{score_str}.score.bar'.lower().replace(' ', '_')
# fig.write_image(f'{file_prefix}.pdf')
# fig.write_image(f'{file_prefix}.png')

### Supplementary Fig. 7-9

In [10]:
def plot_agent_result(score_str):
    model_rank_list = [[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
    with open(f'PhytoBench-Gene-for_plot/score.{score_str}.tsv', 'r') as open_tsv:
        for eachline in open_tsv.readlines()[1:]:
            sp = eachline.strip().split()
            rank_array = np.array([int(s[1]) - 1 for s in sp[-4:]])[[3, 0, 2, 1]]
            for i, r in enumerate(rank_array):
                model_rank_list[i][r] += 1
    num = sum(model_rank_list[0])
    if sum(model_rank_list[1]) == num and sum(model_rank_list[2]) == num and sum(model_rank_list[3]) == num:
        model_rank_array = np.array(model_rank_list) / num
    else:
        raise 'error'

    line_width = 2
    rank_color_list = ['rgb(0,152,210)', 'rgb(140,197,240)', 'rgb(197,224,247)', 'rgb(230,242,250)']
    fig = go.Figure()
    for i in list(range(4)):
        fig.add_trace(go.Bar(
            x=x_list,
            y=model_rank_array[:,i],
            name=f'Rank{i+1}',
            marker_color=rank_color_list[i],
        ))
    fig.update_layout(
        barmode='stack',
        showlegend=False,
        xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
        yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Percent (%)'},
        plot_bgcolor='white',
        font_family='Arial',
        font_color='rgb(0,0,0)',
        font_size=20,
        width=1080,
        height=1080)
    fig.show()
    # file_prefix = f'supplementary_fig.7.phytobench-gene.{score_str}.percent.bar'.lower().replace(' ', '_')
    # fig.write_image(f'{file_prefix}.pdf')
    # fig.write_image(f'{file_prefix}.png')

    results, prob_matrix, _ = calculate_pl_elo(f'PhytoBench-Gene-for_plot/score.{score_str}.tsv', save_csv=False)

    reordered_matrix = prob_matrix[np.ix_([3, 0, 2, 1], [3, 0, 2, 1])]
    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        x=x_list,
        y=x_list,
        z=reordered_matrix,
        colorscale='TroPic',
    ))
    fig.update_layout(
        showlegend=False,
        yaxis_scaleanchor='x',
        xaxis={'showline': False, 'linewidth': line_width, 'linecolor': 'black', 'mirror': False, 'ticks': 'outside', 'tickwidth': line_width},
        yaxis={'showline': False, 'linewidth': line_width, 'linecolor': 'black', 'mirror': False, 'ticks': 'outside', 'tickwidth': line_width},
        plot_bgcolor='white',
        font_family='Arial',
        font_color='rgb(0,0,0)',
        font_size=20,
        width=1080,
        height=1080)
    fig.show()
    # file_prefix = f'supplementary_fig.8.phytobench-gene.{score_str}.prob.heatmap'.lower().replace(' ', '_')
    # fig.write_image(f'{file_prefix}.pdf')
    # fig.write_image(f'{file_prefix}.png')

    y_list = [0, 0, 0, 0]
    for r in results:
        index = model_list.index(r[0])
        y_list[index] = r[1]

    color_list = ['rgb(31,113,179)', 'rgb(208,210,211)', 'rgb(208,210,211)', 'rgb(208,210,211)']
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=x_list,
        y=y_list,
        marker_color=color_list,
        marker_line_color='rgb(0,0,0)',
        marker_line_width=line_width,
        text=[f'{round(num)}' for num in y_list],
        textposition='outside',
    ))
    fig.update_layout(
        showlegend=False,
        xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
        yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Score', 'range': [1000, 2000]},
        plot_bgcolor='white',
        font_family='Arial',
        font_color='rgb(0,0,0)',
        font_size=20,
        width=1080,
        height=1080)
    fig.show()
    # file_prefix = f'supplementary_fig.9.phytobench-gene.{score_str}.score.bar'.lower().replace(' ', '_')
    # fig.write_image(f'{file_prefix}.pdf')
    # fig.write_image(f'{file_prefix}.png')

In [11]:
score_list = [
    'well_studied',
    'well_studied.rice',
    'well_studied.maize',
    'well_studied.wheat',
    'well_studied.soybean',
    'well_studied.arabidopsis',
    'uncharacterized',
    'uncharacterized.rice',
    'uncharacterized.maize',
    'uncharacterized.wheat',
    'uncharacterized.soybean',
    'uncharacterized.arabidopsis',
    'rice',
    'maize',
    'wheat',
    'soybean',
    'arabidopsis',
]

for score_str in score_list:
    plot_agent_result(score_str)

Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 300

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1646.18  95% CI [1610.57, 1681.79]
Gemini      Elo=1584.78  95% CI [1584.78, 1584.78]
OpenAI      Elo=1469.08  95% CI [1433.18, 1504.97]
Grok        Elo=1299.97  95% CI [1257.32, 1342.61]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.837       0.661       0.413
Grok             0.163          --       0.274       0.120
OpenAI           0.339       0.726          --       0.265
Phytomni         0.587       0.880       0.735          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 60

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1661.59  95% CI [1581.00, 1742.18]
Gemini      Elo=1583.19  95% CI [1583.19, 1583.19]
OpenAI      Elo=1413.73  95% CI [1330.66, 1496.80]
Grok        Elo=1341.49  95% CI [1252.72, 1430.27]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.801       0.726       0.389
Grok             0.199          --       0.398       0.137
OpenAI           0.274       0.602          --       0.194
Phytomni         0.611       0.863       0.806          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 60

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1658.07  95% CI [1580.28, 1735.86]
Gemini      Elo=1563.58  95% CI [1563.58, 1563.58]
OpenAI      Elo=1480.89  95% CI [1400.83, 1560.95]
Grok        Elo=1297.45  95% CI [1202.10, 1392.81]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.822       0.617       0.367
Grok             0.178          --       0.258       0.111
OpenAI           0.383       0.742          --       0.265
Phytomni         0.633       0.889       0.735          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 60

=== Elo-like scores from 4-way Plackett–Luce ===
Gemini      Elo=1624.48  95% CI [1624.48, 1624.48]
Phytomni    Elo=1594.58  95% CI [1515.43, 1673.73]
OpenAI      Elo=1493.66  95% CI [1412.57, 1574.75]
Grok        Elo=1287.28  95% CI [1187.90, 1386.66]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.874       0.680       0.543
Grok             0.126          --       0.234       0.146
OpenAI           0.320       0.766          --       0.359
Phytomni         0.457       0.854       0.641          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 60

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1933.69  95% CI [1818.40, 2048.98]
Gemini      Elo=1616.66  95% CI [1616.66, 1616.66]
OpenAI      Elo=1373.87  95% CI [1271.90, 1475.85]
Grok        Elo=1075.77  95% CI [ 936.52, 1215.02]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.957       0.802       0.139
Grok             0.043          --       0.152       0.007
OpenAI           0.198       0.848          --       0.038
Phytomni         0.861       0.993       0.962          --


...

Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 120

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1672.75  95% CI [1615.50, 1729.99]
Gemini      Elo=1565.71  95% CI [1565.71, 1565.71]
OpenAI      Elo=1423.87  95% CI [1365.67, 1482.06]
Grok        Elo=1337.67  95% CI [1274.64, 1400.70]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.788       0.693       0.351
Grok             0.212          --       0.378       0.127
OpenAI           0.307       0.622          --       0.193
Phytomni         0.649       0.873       0.807          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 120

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1659.37  95% CI [1603.54, 1715.20]
Gemini      Elo=1563.65  95% CI [1563.65, 1563.65]
OpenAI      Elo=1470.65  95% CI [1413.78, 1527.52]
Grok        Elo=1306.33  95% CI [1239.81, 1372.84]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.815       0.631       0.366
Grok             0.185          --       0.280       0.116
OpenAI           0.369       0.720          --       0.252
Phytomni         0.634       0.884       0.748          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 120

=== Elo-like scores from 4-way Plackett–Luce ===
Gemini      Elo=1609.21  95% CI [1609.21, 1609.21]
Phytomni    Elo=1597.70  95% CI [1543.05, 1652.35]
OpenAI      Elo=1501.76  95% CI [1445.27, 1558.24]
Grok        Elo=1291.33  95% CI [1220.85, 1361.81]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.862       0.650       0.517
Grok             0.138          --       0.229       0.146
OpenAI           0.350       0.771          --       0.365
Phytomni         0.483       0.854       0.635          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 120

=== Elo-like scores from 4-way Plackett–Luce ===
Phytomni    Elo=1860.88  95% CI [1791.32, 1930.43]
Gemini      Elo=1621.86  95% CI [1621.86, 1621.86]
OpenAI      Elo=1398.44  95% CI [1330.35, 1466.52]
Grok        Elo=1118.82  95% CI [1025.52, 1212.13]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.948       0.783       0.202
Grok             0.052          --       0.167       0.014
OpenAI           0.217       0.833          --       0.065
Phytomni         0.798       0.986       0.935          --


Models: ['Gemini', 'Grok', 'OpenAI', 'Phytomni'] | Reference model fixed at 0: Gemini
Total 4-way battles (rows used): 120

=== Elo-like scores from 4-way Plackett–Luce ===
Gemini      Elo=1568.49  95% CI [1568.49, 1568.49]
Phytomni    Elo=1563.73  95% CI [1507.66, 1619.79]
OpenAI      Elo=1505.00  95% CI [1451.06, 1558.93]
Grok        Elo=1362.78  95% CI [1300.94, 1424.63]

=== Pairwise win probability matrix (p(row > col)) ===
                Gemini        Grok      OpenAI    Phytomni
Gemini              --       0.766       0.590       0.507
Grok             0.234          --       0.306       0.239
OpenAI           0.410       0.694          --       0.416
Phytomni         0.493       0.761       0.584          --
